# Spam Detection using Average Word2Vec and Random Forest

## Problem Statement
Build an NLP model that classifies SMS messages as **Spam** or **Ham** using **Average Word2Vec** embeddings and a **Random Forest** classifier.

## Objectives
- Load SMS Spam dataset
- Clean and preprocess text
- Tokenize messages
- Train Word2Vec embeddings
- Create Average Word2Vec features
- Train Random Forest
- Evaluate performance
- Predict labels for new messages


In [1]:
# Install once if needed
# !pip install gensim tqdm nltk scikit-learn pandas numpy

import re
import numpy as np
import pandas as pd
import nltk
import gensim

from tqdm import tqdm
from nltk.stem import WordNetLemmatizer
from gensim.utils import simple_preprocess
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('wordnet')

messages=pd.read_csv('SMSSpamCollection',sep='\t',names=['label','message'])
print(messages.shape)
messages.head()

lemmatizer=WordNetLemmatizer()
corpus=[]

for text in messages['message']:
    review=re.sub('[^a-zA-Z]',' ',text)
    review=review.lower().split()
    review=[lemmatizer.lemmatize(word) for word in review]
    corpus.append(' '.join(review))

words=[simple_preprocess(text) for text in corpus]

model=gensim.models.Word2Vec(
    sentences=words,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    epochs=10,
    seed=42
)

def avg_word2vec(doc):
    vectors=[model.wv[word] for word in doc if word in model.wv]
    if len(vectors)==0:
        return np.zeros(model.vector_size)
    return np.mean(vectors,axis=0)

X=np.array([avg_word2vec(doc) for doc in tqdm(words)])
print("X shape:",X.shape)

y=pd.get_dummies(messages['label']).iloc[:,0].values
print("y shape:",y.shape)

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y
)

classifier=RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

classifier.fit(X_train,y_train)

y_pred=classifier.predict(X_test)

print("Accuracy:",accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

# Predict new messages
test_messages=[
    "Congratulations! You have won a FREE iPhone. Call now.",
    "Hi, are we meeting at 6 PM today?",
    "URGENT! Claim your cash reward immediately."
]

for msg in test_messages:
    review=re.sub('[^a-zA-Z]',' ',msg)
    review=review.lower().split()
    review=[lemmatizer.lemmatize(word) for word in review]
    review=' '.join(review)
    tokens=simple_preprocess(review)
    vec=avg_word2vec(tokens).reshape(1,-1)
    pred=classifier.predict(vec)[0]
    print(msg)
    print("Prediction:","Spam" if pred==1 else "Ham")
    print("-"*60)


[nltk_data] Downloading package wordnet to
[nltk_data]     c:\DS2026\Python\.venv\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


(5572, 2)


100%|██████████| 5572/5572 [00:00<00:00, 10758.32it/s]


X shape: (5572, 100)
y shape: (5572,)
Accuracy: 0.9695067264573991
              precision    recall  f1-score   support

       False       0.92      0.85      0.88       149
        True       0.98      0.99      0.98       966

    accuracy                           0.97      1115
   macro avg       0.95      0.92      0.93      1115
weighted avg       0.97      0.97      0.97      1115

[[126  23]
 [ 11 955]]
Congratulations! You have won a FREE iPhone. Call now.
Prediction: Ham
------------------------------------------------------------
Hi, are we meeting at 6 PM today?
Prediction: Spam
------------------------------------------------------------
URGENT! Claim your cash reward immediately.
Prediction: Ham
------------------------------------------------------------


## Verification Checklist

- `messages.shape` should be `(5569, 2)`
- `X.shape` should be `(5569, 100)`
- `y.shape` should be `(5569,)`
- `np.isnan(X).sum()` should be `0`
- Model should train without errors.
- Test predictions should classify unseen SMS messages.


In [2]:
index = 20

print(messages['message'][index])

prediction = classifier.predict(X[index].reshape(1, -1))

print("Prediction:", "Spam" if prediction[0] == 1 else "Ham")
print("Actual:", messages['label'][index])

Is that seriously how you spell his name?
Prediction: Spam
Actual: ham


In [4]:
import random

indices = random.sample(range(len(X_test)), 20)

for idx in indices:
    pred = classifier.predict(X_test[idx].reshape(1, -1))[0]

    original_index = y_test